In [1]:
!pip install ucimlrepo

In [3]:
!pip install scikit-learn

In [9]:
import sys
!{sys.executable} -m pip uninstall -y numpy scipy scikit-learn
!{sys.executable} -m pip install --no-cache-dir "numpy==1.26.4" "scipy==1.14.1" "scikit-learn==1.5.2"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.14.1
Uninstalling scipy-1.14.1:
  Successfully uninstalled scipy-1.14.1
Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 278.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 293.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 341.4 MB/s eta 0:00:00


In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

dry_bean = fetch_ucirepo(id=602)

df = dry_bean.data.features.copy()
df["Class"] = dry_bean.data.targets.to_numpy().ravel()

print(df.shape)
print(df.columns.tolist())
print(df["Class"].value_counts())

df.to_csv("dry_bean.csv", index=False)
print("saved dry_bean.csv")

(13611, 17)
['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'Roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64
saved dry_bean.csv


In [1]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)

df = pd.read_csv("dry_bean.csv")
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

test_df = X_test.copy()
test_df["Class"] = y_test.to_numpy()
test_df.to_csv("test_data.csv", index=False)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=42)),
    ]),
}

os.makedirs("model", exist_ok=True)
rows = []

for name, model in models.items():
    print("training", name)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)
    else:
        y_score = model.decision_function(X_test)

    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score, multi_class="ovr", average="weighted"),
        "Precision": precision_score(y_test, y_pred, average="weighted"),
        "Recall": recall_score(y_test, y_pred, average="weighted"),
        "F1": f1_score(y_test, y_pred, average="weighted"),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })

    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, os.path.join("model", filename))
    print("saved", filename)

results = pd.DataFrame(rows)
results.to_csv("model_results.csv", index=False)
print(results.round(4).to_string(index=False))

training Logistic Regression
saved logistic_regression.joblib
training Decision Tree
saved decision_tree.joblib
training KNN
saved knn.joblib
training Naive Bayes
saved naive_bayes.joblib
training Support Vector Machine


ValueError: Target scores need to be probabilities for multiclass roc_auc, i.e. they should sum up to 1.0 over classes

In [3]:
from scipy.special import softmax

y_pred = model.predict(X_test)
y_score = softmax(model.decision_function(X_test), axis=1)

rows.append({
    "ML Model Name": "Support Vector Machine",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_score, multi_class="ovr", average="weighted"),
    "Precision": precision_score(y_test, y_pred, average="weighted"),
    "Recall": recall_score(y_test, y_pred, average="weighted"),
    "F1": f1_score(y_test, y_pred, average="weighted"),
    "MCC": matthews_corrcoef(y_test, y_pred),
})

joblib.dump(model, os.path.join("model", "support_vector_machine.joblib"))
print("saved support_vector_machine.joblib")

results = pd.DataFrame(rows)
results.to_csv("model_results.csv", index=False)
print(results.round(4).to_string(index=False))

saved support_vector_machine.joblib
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9207 0.9934     0.9215  0.9207 0.9209 0.9042
         Decision Tree    0.8917 0.9330     0.8912  0.8917 0.8913 0.8691
                   KNN    0.9166 0.9812     0.9174  0.9166 0.9168 0.8992
           Naive Bayes    0.7639 0.9644     0.7654  0.7639 0.7615 0.7154
Support Vector Machine    0.9221 0.9884     0.9224  0.9221 0.9222 0.9059


In [5]:
import os
print(os.getcwd())
print(os.listdir("model"))

/home/s2026aj05938
['logistic_regression.joblib', 'decision_tree.joblib', 'knn.joblib', 'naive_bayes.joblib', 'support_vector_machine.joblib']


In [7]:
%%writefile app.py
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

FEATURES = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength", "AspectRatio",
    "Eccentricity", "ConvexArea", "EquivDiameter", "Extent", "Solidity",
    "Roundness", "Compactness", "ShapeFactor1", "ShapeFactor2",
    "ShapeFactor3", "ShapeFactor4",
]
TARGET = "Class"

MODEL_FILES = {
    "Logistic Regression": "model/logistic_regression.joblib",
    "Decision Tree": "model/decision_tree.joblib",
    "KNN": "model/knn.joblib",
    "Naive Bayes": "model/naive_bayes.joblib",
    "Support Vector Machine": "model/support_vector_machine.joblib",
}

st.set_page_config(page_title="Dry Bean Classifier", layout="wide")
st.title("Dry Bean Type Classification")
st.write(
    "Seven dry-bean varieties are predicted from 16 size and shape measurements. "
    "Upload the test CSV. The chosen saved model is loaded and scored. It is not retrained."
)

model_name = st.selectbox("Choose a model", list(MODEL_FILES.keys()))
uploaded = st.file_uploader("Upload test CSV", type=["csv"])

if uploaded is not None:
    data = pd.read_csv(uploaded)
    missing = [column for column in FEATURES + [TARGET] if column not in data.columns]
    if missing:
        st.error("These columns are missing: " + ", ".join(missing))
    else:
        model = joblib.load(MODEL_FILES[model_name])
        X = data[FEATURES]
        y_true = data[TARGET]
        y_pred = model.predict(X)

        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X)
        else:
            y_score = softmax(model.decision_function(X), axis=1)

        accuracy = accuracy_score(y_true, y_pred)
        auc = roc_auc_score(y_true, y_score, multi_class="ovr", average="weighted")
        precision = precision_score(y_true, y_pred, average="weighted")
        recall = recall_score(y_true, y_pred, average="weighted")
        f1 = f1_score(y_true, y_pred, average="weighted")
        mcc = matthews_corrcoef(y_true, y_pred)

        st.subheader(model_name)
        c1, c2, c3 = st.columns(3)
        c1.metric("Accuracy", f"{accuracy:.4f}")
        c2.metric("AUC", f"{auc:.4f}")
        c3.metric("Precision", f"{precision:.4f}")
        c4, c5, c6 = st.columns(3)
        c4.metric("Recall", f"{recall:.4f}")
        c5.metric("F1", f"{f1:.4f}")
        c6.metric("MCC", f"{mcc:.4f}")

        labels = sorted(y_true.unique())
        matrix = confusion_matrix(y_true, y_pred, labels=labels)
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=ax,
        )
        ax.set_xlabel("Predicted bean type")
        ax.set_ylabel("Actual bean type")
        st.subheader("Confusion matrix")
        st.pyplot(fig)
        plt.close(fig)

        st.subheader("Classification report")
        st.text(classification_report(y_true, y_pred))

Writing app.py


In [9]:
%%writefile requirements.txt
streamlit
scikit-learn==1.5.2
numpy==1.26.4
pandas==2.2.3
matplotlib
seaborn
scipy==1.14.1
joblib

Writing requirements.txt


In [11]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 166.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 143.3 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.36.2

In [1]:
%%writefile README.md
# Dry Bean Type Classification

## a. Problem statement

Dry beans of seven registered varieties look similar, so they are hard to separate by eye. This project classifies each bean into one variety using 16 size and shape measurements taken from images. The varieties are BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, SEKER, and SIRA.

Five saved classifiers are compared on the same held-out test set. A Streamlit app loads those saved models and scores an uploaded test CSV. The app does not retrain the models.

## b. Dataset description

- Source: UCI Machine Learning Repository, Dry Bean Dataset, id 602
- https://archive.ics.uci.edu/dataset/602/dry+bean+dataset
- Rows: 13,611 beans
- Features: 16 numeric measurements, with no missing values
- Target: Class (7 varieties)
- Class counts: DERMASON 3546, SIRA 2636, SEKER 2027, HOROZ 1928, CALI 1630, BARBUNYA 1322, BOMBAY 522
- Split: 80% train and 20% test, stratified by class, random_state 42
- Test file used by the app: test_data.csv

Feature columns: Area, Perimeter, MajorAxisLength, MinorAxisLength, AspectRatio, Eccentricity, ConvexArea, EquivDiameter, Extent, Solidity, Roundness, Compactness, ShapeFactor1, ShapeFactor2, ShapeFactor3, ShapeFactor4.

Logistic Regression, KNN, and SVM use a saved StandardScaler inside the model pipeline. Decision Tree and Gaussian Naive Bayes use the raw measurements.

## c. Github Repository Link

Add the public GitHub repository link here after the repository is created.

## d. Models used

Precision, Recall, and F1 use a weighted average because the seven classes are unequal in size. AUC is weighted one-vs-rest.

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
| --- | --- | --- | --- | --- | --- | --- |
| Logistic Regression | 0.9207 | 0.9934 | 0.9215 | 0.9207 | 0.9209 | 0.9042 |
| Decision Tree | 0.8917 | 0.9330 | 0.8912 | 0.8917 | 0.8913 | 0.8691 |
| KNN | 0.9166 | 0.9812 | 0.9174 | 0.9166 | 0.9168 | 0.8992 |
| Naive Bayes | 0.7639 | 0.9644 | 0.7654 | 0.7639 | 0.7615 | 0.7154 |
| Support Vector Machine | 0.9221 | 0.9884 | 0.9224 | 0.9221 | 0.9222 | 0.9059 |

## e. Observations

| ML Model Name | Observation about model performance |
| --- | --- |
| Logistic Regression | Very strong on every metric. Accuracy is 0.9207, and it has the highest AUC at 0.9934, so it separates the seven varieties very well. |
| Decision Tree | Accuracy is 0.8917. It is usable, but it is behind Logistic Regression, KNN, and SVM on every score. A single tree does not generalize as well on this data. |
| KNN | Accuracy is 0.9166 and AUC is 0.9812. It is close to the top two models. Predictions depend on nearby beans in the scaled feature space. |
| Naive Bayes | Weakest classifier. Accuracy is only 0.7639 and MCC is 0.7154. AUC is still high at 0.9644, so the score ranking is better than the final class decision. Gaussian Naive Bayes assumes the features are independent, and these shape measurements are related to each other. |
| Support Vector Machine | Best accuracy (0.9221), best F1 (0.9222), and best MCC (0.9059). AUC is 0.9884, slightly below Logistic Regression. |
| Overall, Winner for your dataset? | Support Vector Machine. It gives the most correct variety labels on the test set. Logistic Regression is a very close second and is the best model if AUC is the main score. |

Writing README.md
